In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

columns = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty'
]

train = pd.read_csv('../data/KDDTrain+.txt', names=columns)
test = pd.read_csv('../data/KDDTest+.txt', names=columns)

# Drop difficulty column - not a feature, just metadata
train = train.drop(columns=['difficulty'])
test = test.drop(columns=['difficulty'])

In [2]:
cat_cols = ['protocol_type', 'service', 'flag']
le = LabelEncoder()

for col in cat_cols:
    combined = pd.concat([train[col], test[col]])  # fit on both to avoid unseen labels
    le.fit(combined)
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])

print("Encoding done")
train[['protocol_type','service','flag']].head()

Encoding done


,protocol_type,service,flag
0,1,20,9
1,2,44,9
2,1,49,5
3,1,24,9
4,1,24,9


In [3]:
train['label'] = train['label'].apply(lambda x: 0 if x == 'normal' else 1)
test['label'] = test['label'].apply(lambda x: 0 if x == 'normal' else 1)

print("Label distribution after encoding:")
print(train['label'].value_counts())

Label distribution after encoding:
label
0    67343
1    58630
Name: count, dtype: int64


In [4]:
X_train = train.drop(columns=['label'])
y_train = train['label']

X_test = test.drop(columns=['label'])
y_test = test['label']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train features shape:", X_train_scaled.shape)
print("Test features shape:", X_test_scaled.shape)

Train features shape: (125973, 41)
Test features shape: (22544, 41)


In [5]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", pd.Series(y_train).value_counts().to_dict())
print("After SMOTE: ", pd.Series(y_train_res).value_counts().to_dict())

Before SMOTE: {0: 67343, 1: 58630}
After SMOTE:  {0: 67343, 1: 67343}
